In [ ]:
"""Train the DeepSVDD detector and scale its anomaly scores."""

import os
import sys

# Clear any cached modules
if 'config' in sys.modules:
    del sys.modules['config']
if 'to_tflite' in sys.modules:
    del sys.modules['to_tflite']

import pandas as pd
from pyod.models.deep_svdd import DeepSVDD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import config
import to_tflite_for_deepsvdd

# Build the detector from its configuration.
deepsvdd = DeepSVDD(
    n_features=len(config.FEATURES),
    contamination=config.CONTAMINATION,
    use_ae=config.USE_AE,
    hidden_neurons=config.HIDDEN_NEURONS,
    hidden_activation=config.HIDDEN_ACTIVATION,
    output_activation=config.OUTPUT_ACTIVATION,
    optimizer=config.OPTIMIZER,
    epochs=config.DSVDD_EPOCHS,
    batch_size=config.DSVDD_BATCH_SIZE,
    dropout_rate=config.DROPOUT_RATE,
    l2_regularizer=config.L2_REGULARIZER,
    random_state=config.RANDOM_STATE,
    verbose=config.VERBOSE,
)

# NOTE: all dataset features must be pre-normalized so that, after the int8
# quantization in keras/to_tflite.py, no value overflows the int8 range.
# Fit the configured features from the CSV.
features = pd.read_csv(config.CSV_PATH)[config.FEATURES].to_numpy()
X_train, X_test = train_test_split(
    features, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)
deepsvdd.fit(X_train)

# Scale the raw anomaly scores into the configured range.
raw_scores = deepsvdd.decision_function(X_train).reshape(-1, 1)
score_scaler = MinMaxScaler(feature_range=config.SCORE_RANGE).fit(raw_scores)
train_scores = score_scaler.transform(raw_scores).ravel()
test_scores = score_scaler.transform(
    deepsvdd.decision_function(X_test).reshape(-1, 1)
).ravel()

# Learn the scores with a small network and export .keras + .tflite.
to_tflite_for_deepsvdd.to_tflite_deep_svdd(
    name="deepsvdd",
    x_train=X_train,
    y_train=train_scores,
    x_test=X_test,
    y_test=test_scores,
    layers=config.LAYERS,
    latent_dim=config.LATENT_DIM,
    learning_rate=config.LEARNING_RATE,
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    verbose=config.VERBOSE,
    feature_names=config.FEATURES,
)


Epoch 1/100, Loss: 2305.3896078169346
Epoch 2/100, Loss: 2305.389612287283
Epoch 3/100, Loss: 2305.389608055353
Epoch 4/100, Loss: 2305.389610260725
Epoch 5/100, Loss: 2305.3896128833294
Epoch 6/100, Loss: 2305.3896076232195
Epoch 7/100, Loss: 2305.3896109759808
Epoch 8/100, Loss: 2305.3896090090275
Epoch 9/100, Loss: 2305.389609143138
Epoch 10/100, Loss: 2305.3896085321903
Epoch 11/100, Loss: 2305.3896087110043
Epoch 12/100, Loss: 2305.389612734318
Epoch 13/100, Loss: 2305.389610990882
Epoch 14/100, Loss: 2305.3896092772484
Epoch 15/100, Loss: 2305.3896124362946
Epoch 16/100, Loss: 2305.389606654644
Epoch 17/100, Loss: 2305.389610081911
Epoch 18/100, Loss: 2305.3896118998528
Epoch 19/100, Loss: 2305.38960903883
Epoch 20/100, Loss: 2305.3896115124226
Epoch 21/100, Loss: 2305.389613389969
Epoch 22/100, Loss: 2305.3896052241325
Epoch 23/100, Loss: 2305.389608681202
Epoch 24/100, Loss: 2305.389612108469
Epoch 25/100, Loss: 2305.389610797167
Epoch 26/100, Loss: 2305.389607310295
Epoch 27/1

2026-08-14 18:58:01.148050: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 758us/step - loss: 0.0014 - val_loss: 9.9094e-05
Epoch 2/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 743us/step - loss: 7.9909e-05 - val_loss: 6.8626e-05
Epoch 3/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 735us/step - loss: 5.9815e-05 - val_loss: 4.6721e-05
Epoch 4/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 708us/step - loss: 4.9040e-05 - val_loss: 5.3047e-05
Epoch 5/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 729us/step - loss: 4.5232e-05 - val_loss: 4.2184e-05
Epoch 6/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 713us/step - loss: 4.3388e-05 - val_loss: 4.7817e-05
Epoch 7/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 730us/step - loss: 3.9332e-05 - val_loss: 3.5253e-05
Epoch 8/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 739us/step - loss: 3.8606e-05 - val_loss: 3.4329e-05
Epoch 9/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 748us/step - loss: 3.8319e-05 - val_loss: 3.5279e-05
Epoch 10/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 743us/step - loss: 3.7625e-05 - val_loss: 3.8969e-05
Epoch 11/50
3747/3747 ━━━━━━━━━━━━

INFO:tensorflow:Assets written to: /tmp/tmpeqad99t4/assets


Saved artifact at '/tmp/tmpeqad99t4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139827578999376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827579001872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827579002832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827579001104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827579003024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139827579001296: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/patrik/Diplomka/LODA/env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1786726819.149216   25285 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786726819.149226   25285 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-08-14 19:00:19.149396: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpeqad99t4
2026-08-14 19:00:19.149715: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-08-14 19:00:19.149718: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpeqad99t4
I0000 00:00:1786726819.151654   25285 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-08-14 19:00:19.151931: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-08-14 1

<Sequential name=sequential, built=True>